# Full AIDE — test-only trên đúng cohort CommFor cũ

Notebook này **không train lại**. Nó nạp checkpoint Full AIDE tốt nhất đã train trên Tiny-GenImage combined và dự đoán đúng cohort CommFor đã dùng để so sánh AIDE forensic với NPR-ResNet18:

- 100 ảnh real dùng chung;
- 9 generator × 100 ảnh fake = 900 ảnh fake;
- tổng cộng 1.000 ảnh;
- threshold cố định `0.5`, không tune trên CommFor.

Hai SHA-256 được khóa cứng để tránh test nhầm dữ liệu hoặc nhầm weight:

- checkpoint: `62848895b44255d6a0567754c503a05807ed0a29b8046f3c6956a049844e47c3`;
- manifest: `20b04a56d2709d091d51fe585621686b2cd792a373a20ec5068fc6a5be28b9cd`.

## Chuẩn bị trên Kaggle

1. Bật **GPU accelerator**.
2. Tạo Kaggle Dataset từ thư mục `baselines/aide_original_full/artifacts/checkpoints` và attach dataset đó vào notebook. Notebook tự tìm `model_trainable.pt` bằng checksum.
3. Bật Internet để tải `OwensLab/CommunityForensics-Eval` và OpenCLIP ConvNeXt-XXLarge. Nếu đã upload `open_clip_pytorch_model.bin` vào Kaggle Input thì notebook sẽ dùng file local.
4. Nếu dùng Hugging Face thường xuyên, nên tạo Kaggle Secret `HF_TOKEN`.

Kết quả cuối nằm tại `/kaggle/working/aide_original_full_commfor_exact_eval` và file ZIP tải về nằm tại `/kaggle/working/aide_original_full_commfor_exact_eval.zip`.

In [ ]:
%pip install -q "open_clip_torch==2.26.1" "datasets>=2.19,<4" "pandas<3.0" "scikit-learn<1.9" "pillow<12.0" "tqdm"

In [ ]:
import os
from pathlib import Path

import torch

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    print("HF_TOKEN not configured; public downloads will be anonymous.")

assert torch.cuda.is_available(), "Kaggle Settings → Accelerator → GPU trước khi chạy."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
CONFIG = {
    "input_root": "/kaggle/input",
    "output_root": "/kaggle/working/aide_original_full_commfor_exact_eval",

    # None = tự tìm đúng model_trainable.pt bằng SHA-256 trong /kaggle/input.
    "checkpoint_path": None,

    # None = tự tìm open_clip_pytorch_model.bin; nếu không có sẽ tải từ HF.
    "semantic_checkpoint": None,

    "commfor_dataset_name": "OwensLab/CommunityForensics-Eval",
    "commfor_split": "CompEval",
    "commfor_streaming": True,
    "commfor_shuffle_seed": 43,
    "commfor_shuffle_buffer_size": 1000,
    "max_scan_records": 500000,

    "threshold": 0.5,
    "image_size": 256,
    "random_seed": 42,

    # Lưu predictions từng phần; chạy lại cell sẽ bỏ qua các ảnh đã hoàn thành.
    "resume": True,
    "save_every": 25,
}

CONFIG

In [ ]:
import base64
import gzip
import importlib.util
import sys
from pathlib import Path

TRAIN_RUNTIME_GZIP_BASE64 = """..."""
TEST_RUNTIME_GZIP_BASE64 = """H4sIAAAAAAACCsU8a2/bSJLf9Su4PNwtOSMzsmI7iRYawOPYGWMTJ3CcxeB8AkGRLZlritTwYUfj9X+/qupu9oOUbM/s4Qa7sdSP6urq6nq3XNc9zxesZHnM9oo82zhnTZY5x+fvTx12F2VNVKdF7sD/6hvmsKqO5lla3bDEOSlWq7OidOLipijrYDC4ukkrp2zynJVOVkRJRVPmMEeDGd+w+HZdpHnt1GWU5gAIYF+l+WbvA8vPV9GSOVGeDMTajANh36O4dvaHo9FoL6UxuHqTp/UGUGB5lcaVQMRZl+wuLZoKttJUAH6+GSAIXH1vIQbjEs7Fl8u9S1ZdsHr/LaywZmW6YnldTQaDH5z90cipbqISAJQsyhxatfqb6FlEt0w0OYuyWDksim+cYuHksCNnyYAEUV2UFVGF6ZvGhVdRni6QLADemRf1DeBc1CyuCVvn6y/He+PDIz6rCpyW0mk1qOoyjWvYGlCmpvOaAIGA7os0Yw5sAXa4qpy84NRN8+UQ+ktW3RRZ4tQNbwFYqyJh2aBiGSwLBxwMXNcdDGgvYbho6qZkYQhbXCNJozwvamKEajAQbXGR1+x7naVz2bKM5aebqLrROtJCfvpnVeTyc3XT1GnGl1xHNU6Q632Br7yj3qwBY9l+nG/a9fNmtd44Eex1LZvWQFxogP+tE9kGpxDfGF+CPA8WTU7bhoOF0Wd8rSSqo4rVlVwNeTgUjXzEl/OPspM4dcj/nAHtBbq/JasgaupCDsOGFmU6kjBKExYWZbpMYXkgdZaFNbB/GMMpw+mFt9FyCWeJG4GRg8GgXSP4+Pn4fXh1+e3i5Pjq9H14/un4w+lXZ+pclQ0MPP31y+kJtp/8cnry9y+fzy+uQuAlZKWp4w0c+M89Gr89ePv23eH84GB8eJgcRaPDozdvDg/iw9Fr+Px29IYlo2j8bv52dHC0eB0fvTuEMQfv3h4csIM38Wt34Kt1Ph1fnJ+dfu2uMh7NRwfR4VEyfjN6l4ze7SeH+wt2+PbwaLx/9PZoPo6TN+/G0es3r6PxiMHaR28X8VF0OGfjt/N3cYKr4H0Nzz5fnl58PT8JTz7/8vkSFvr87fLkNAQKhOfvYT1YaXw0ejd+E45G+0f7h65C7sPpxenl8dXny68Kr1+KplxmUVW5Q97wKU3+CW052/zjMBzL1rO0ZItsExLhX/e2jnsgHIX7svX92YfjC/nlw/HH8y/yyy3waJpXt5twrBa8vUtv8Trldbhq4YV3R9DvDwbvT8+Ov328AhpcnJ1/mDgJSIBrEARDvA8z2N0Dh5Lm66YOy6Ko3YnjvuJ89Ipa5UJFU/eNuS/KW7hmr3pYU3IlSeAQpbKEpYRaiNcX4F0UOROdUsT1dFUMOus0DhWALQNIRoV5tGKILAicuxwkTvj9exaVS+baw0HwC5US1tESp2QR3PHxPKxeH8zD+dvxbRg1y5Itw6po1p35N1G2QCBxWsE0mI/XSm5WkEGIgxapz/egTz5G81ddfbR3qhNLzK/WWUqkh/Hr3gE1qJsVHEb/+iA0F4uMhRVjCYw4eL2lfw5/GHxNf0c0QWWN2oP5HlZxlIewz6JMKug9HI1Ufwn8Waxa+GPR3OoQaBwFh6I1AZa4T2HCvVzp9VjrQmWV0PpHWuuyjIDL5rAOLi57SJlKKCBMJDqsaojSGi2q6I4BJ7JyQ0OHg0cQkglboL6GmSGqQg/5bkKKxHf2fnIAlQnNTtIlat6p1FEBn+T51HufojKGSUGxZrnnlnPXR0l8A9hmjEPA/xZo99w0+a2T5k5as9LLotU8iSZiZACHmHj7o/GBgwbD+MAfOnPX9RUEhUvQrIGrmEfwOBrAyCACZP8N+84/AZJio0gB1KXaNoeA9wZV1gTFAm0a7xRfkba0BnMjr4PVbZKWHv9STYmuYP2kcFeLW/raT4p7F4blcQHSazl1m3qx97afNohXkIB29gRCQzEE51doV0RVnKbTsyiroA3YBxCZjuXe4owBd6L1xzz6V+2nPcR04VAXWES0STRoPLi1OZinYMfyeUNnAavXPplc+TpIqzzKeZevHYSgNVg/2jdYSYwUWCVpFRfAckIOEovRBLAbszRO6wnOcf6lyTEljqmPN+JElB1aE1qeaPqFnBNFD20Yz7XdcbtSi3qMuiQhI3nqXONgTw7yZxw2kHjL+AosEpZ4NEuh6gegIIu5J/H0fbk8WIAaAI1+UVoxB82Ti6I+K5o8OS3LovQMTl+4J2RDwv7hMB4k9MfAOa5rNJ3Tmgwe5+/c+HnPxSyeK/5JwSiTO+M86bbgOYKrqAZ1Uk0ccE5qIgUqxmtOhWJesfKOJaK3btYZ4woU/plpA+layz3i1e7bsE0LYCzOD9btRgM5zcEwM687kl6TUy0Yvx0n0Q2iNdy8xPOQG9W4oQDk+zpKQlZkxT0II9+ZTm2+kl0mkoJucikLG7FVSVzrzC8b2OCK9R73RaGdsgQAt+u3JkWXSro4DxaSwBCf28N6kIR4tE+buy0AZypBX49mEmNY1BOtvvOTs6/QXpdga3gL91OT1SnwADhbNdtLUf6kMbgCGsb4CZwqZOe/gRuJPsiDXPXRNaS0bBaSorV8btnGK4t7216ji62xYBAEs4kOT5ESzx0xBijXYMvMWebOfH+o7rMmKmlM63jCuB3DuK4lE2bnOHR80VQq4ycGRuBWpejAgnRvR0rR2SIVoosk7A6P/+klTivmNWHMhwdLVnuuZhb6KCH0PgMT6nVBq+bFfe5KfPhwfjzbkeg9IToDYDo6FG1VfjbiQrb7Rf+EhxBCpKRLzMkhTJ0RyeadxPG3cwWB0U6kBbPlkDRctcPfykvacJ0HnjPePACLFfjaz2QD8rMC+lfoQJqNFjhQtrsl7AAbpdPhKh2mmQhqzlBfqWsYqIEBeiCsBLiXH352nwLqoXCphiRjorKMNj1GB1+Y7Ku0CH7GCeefNSi+/7I1kYq+oak07IkihBRcjLQiya4sxBdidi0gzfpR3LI6OYMvW1xbkWbPXkgS1PjPJTxtySB/72Jc+11t1kL1LdxvYNeu12RQqWAdhSlrGAWaDP/ogB9baQQ3iBRuKPWGJz9MnHUSoCF0VqIhhleCzBe0WaR8FOpUhgCEPytieuBpKTdXeE4rkGohmtzf9XYuv7QGpUi0Rk1yaK26gNCaDTnA2x+5pZZWpE5bC1TuIkjShQhEtzQA8mfNKq/UFRbTbVvkHyiI5HHII2jjrMBwclkBEM5EtDwqFuKWA5/jO3+Zkt+8c6VTYb7w0LRaELQiXP0lMPmDAVRbDHVI29Gqd1INfgBOr+dLFHZgoBAgvyTbdOLW/Mir56y7r6/77oXrvpNRcXNBFQpvo3Dc7BTnmxXxdT8yQ50LZ0FSFuswadAHQHPc84O6wOsgHPd2e9qCsAsa0RMS9HfszbJkP7TqHNQKK1/FPFDncG4V4X87N8JTEcBkCp2OERuDcVlXZMI+SYxZQHd5vvE0osBxpb8zQYB5lIQtxAfuYuJR8zYSv9g4BO7yyc/Bb+jiaAQD8vUMF1z4KCmsVtp9NTBAoxlDq6biSYMIFt23uAUopcDKSwIWGjm2tkXN8YcPiH5LvLoIUf2hxcIDW77hEYAb6SFAzt3YQt928fgneZvjYrUuKpBmHCXM2qD+avL0t4aZzoCippDv65JhqIVH/cnP9pQxM9FNj6Hm2hTlhKL/AUbB359cfUF/5qvoGmrWEHIAHbSIGHCzlec5rlgOInZo2VZCd5BJAtTN10FU0ReunyyFB7NRd01hGEVTXo9RJa434GK+csaHh8GIoNW0FIDjK5OFRwkaj5s+wZqVq6Zm3hg4aoiShrzjZVM0lWDgNXfZhk6rv1BFiC17fAFjJHSfBbBzVq6LDANoys8ToJTJDFSaeopiQ416mk2LjsXUnadZmoPJpmkz0NHLHLizhMOtROBKdcJGYEAkYmmaOCqj++1I8h0FTV4BEzG4xyP//x1hRVlPftzjfEippovTq/DT6fEFnr3Z+vXqvbZjD/+8cCK/Ppx9QMJ71y0/ALAZMvFqChRS3CFDoeC4R6UWKRw6P2BQsWZLBpdoXhTosBEF6IJgDvZfPDLYRtb4qmkV8jMRkT8lGnisccr/BgnDiBUooHjdwL8gFlaeIQNQ7HIQQt8iKtzdo3XNyCIKlwbtP4ap3cpblCS7ddtPS+FOZFATtmLea2kSVk2GMSYicy9sDdq0/eRrs6/dFfp2KJ/DdVnMI2SweoOqSGyA4Fy73REBzvR8gxwcqNgt5VRBICIHYq4F94BpZU/wn+hQUe2Bbi5YZGlxRoLhSQshqA+a6IEsPhajAvhNX60viCfw16G1sopmAj2gE0P9YVzdeTq81h4RjQFbretNFzbv7VjzKn9GRge4tdJGN2xz0yZ3ugfyApNbIvqHLO4vgiDaiZKafI7VLRa+tvcMNp809xK4Z1G+8Xaq6z4cpL3RQnLMRfiVlsZq2zffiG7NOMOsHzlOno2o38aOhU0iSdk1SRT+En4b37N3bzjSfDQyrWbytGgGhJflSj/nkDAJBhDJB7UoM32gP4+aQy9jtRpVFApou9KnmZkko4NHrL2nOdiKTaeLTqSRg5uRDaf3SdTaASakp+18Ze/3UAl4lgLKTlQ7BnGGYn/TB/5Xs/FNW19Gni9RAOGFoOwa+YaCWfxHUNooSjakETKGhNZY2TQzJYfhfeYkqLpsCZJJMS36TyK7J8KgTQ5d8vbDRVmkyxBTXJizrTpVBjyzBUePf7aqHw4GRhHfm4UL7YXvrDXQUibQ0yZErXFGTkJYCk0SoeiO7qIU2MuS390shXvsnHx7f+x8+PINgwKtHHROc5ytElA4IIpjMDaIUQOM15DthGoVk+M8AV3fwGF63M9CZK+N/PlMCNGE3aUxa61j/tVzEXl5rHAYSYrygPJxEpherjFTYqZq5jCYDOMNv1qiLAHvk9D1+FFnH/oKpMwxQqbfNE+u/cqA6z8nT8zP3KwDocPfkizVzpjigXYJiW4BY85LkEErbNHTECIRQDUfeHrButZM4O3VULrFa9SpvAR1s8DlhYjLlFEo3N8AzIc+1K0Cqz7ELRvEQEu6asYEUBYV+nejodO1MtAbBA5eMs8IW0kvEA7K74nrbI9hWkuDQkQkFce1jIufZQ1LixWWb3LSUJXAd+7IdNiuvVhoXHoWTw1h9XUI2onKF6cuGO0A7p6ly5u6CrGAUkLVhIsWS1bgRHidckoW54UwGLaP/a7U0mqiLZFU5FiF0SKtRLYFu0dg9Z2K6xyVG2U/qe7eFNhf9LibSwtgYZdzs5mDVJ04n9csP/l4/sU5Acf/gv1a7/3660esr3J+dN6fXL36evkJzCdQiLxU9nDEdZy/0x78lrfGgl77qmEGpqCF+F/17r+2oVLtPFvtYm+Zd8C5Pjwacc7OXD1fRLVGQ6w14jEhXYprI2Y7jc62dlpz0J2kYJVKnFNwUFbjaqhLlcJ5ydScpGb61Kc+WOrJFr0Hw/ow6wH7RNLQHG+WfvUqNGtGt/5LzrJ7+ma25WH6JNHYN94sHNMn6T32TKOwrO98rfH91YdyYl/vVgidgsQOFGvEVkhGqaTOyn0j/G1QOgWOHXysERY+/+FcSVY2L3bJ1lkUA9OTUSQkBRqpQJ4anAPn+K4ADyPKLXD4VgCmVSDPuAi/o3V5VBRA7HE5DRfqPkfhjgZ4a7MGPeecg70LVm9OZi8VWlqxL1lUiCO21J+qVJVuu7s/N2mGtW+aoMbCMoAFmGM7XvNFWfzOciVTa1B1t0EQuNrtxRhWe/zSIpMRmzmuwpnL0+/6UBiRvrJCKZSiVBAfTRrDE+uoDV7v0lgzDbkAK3y9juSFY4xE4l3LNIraXrJ4TK1rGDtU2QPjnrLIaDRbFzECNcU3Z3beN3T29vVKBLdYg4EPVxmLZ9l6y1xrUAcIPlNBTwqkSIYHkoRg/jdlFGORKY96mdVdFvwd80UhoufCSbta1Za+PGecsOWMeRTfzoElgY3jrElIImOEazcST0MZOpq1YyPRmYeH2+VVPZurKlE1o05zNMiuU1jiSHfYx1m+BU7p3O1w+cMjDnObErQL0ZFZTSN5uGVCl297zXFFPcqBh+RX43E9/YDBApA3K+kQGDXbxgCqJKLenk6M/0Hnu06nMtehW32xt06uY8jpikXqWnUSRRazpgr1iiZNYOq+CcgR7Yi5vxBjYBG9Hk+YPTK1IyVfX6ZLcYFmS0x1Q61jZ5ieGADuDJcWhl4+peyHznDDtvB7fUcR3tOjhd3QXyfCeMsw//bQyW5OKIzYk+KU6zx2YuVYD6VfE+X2636V1h600V7wrqyQ9u4Qvb30sJPTH2pBbgy3T0luteYkL+g3KdlGvLgjatYKYOZ2e4w4qnhtDXimbUWAL/04I/jOMzAITS8ZKEG+kWUeinTzQ2dD7ckYkc0th2OUP22N7bZOorVxTbIqyodYTYJ+/taw8o79Xs/EdYujPKfY7Uh6SObWlZcjfHLJCuKrqXn0KEfv25iZJVfp4cu0bxp/EtMZL9/BmOzTfSajT/R79iA+BeJljLUN8GqMG9/7zsbGTXtcs3Oy/ginH03QaEu4EXi6+FLQRK4u6iibmoEYM0AOJm4qhljsYo1MWBVPXWW5go3Nn9Saj3jdPhwpVC1SpNL2JmvTriVftLW4yNqC7t1AvOTEH6fOfqeTC8VOia7fGQj8i2PFPdrGytYMufRPprffeRPVlz9oDx8Y77a3t1Pw3xEkKDOmthK4hv/Puvuj8sHp1pLZLkW4hU/+vVlEsa0EpXfJYW9XZZSe9E7sXgcjiDLckhnZsgMkk/pmVEmA/PO4MzSE48/DeVbE+HpRe7uk/wcGlaFOzBuGXI3vdWPQIwJqSKFOHp2XgU9R1oJm/P6Rv429+PwAZ2DVFgfRO5hEs3pBHeRwM8VXz38GpehOws762TQrlimvLNP8SCJkF5KWLW7T+3y3VbGo4WZ4HBrfvOfzSgxwnq6xjmfGiyC6YPX0ps7+u/KcCiVDlMm3Kb1bfdh6T+2FJhyn4fYJduHsxEgxmruwx/YxeH/Z7aSHJsaAZ4FCB2MhFc6EpyMno6Pk0d0xmSdg+xCQj0uemty+eyVPQ9o4/bA4k7uWo9CBq5LBu+jd/7KlA0xLJk9e+gKiy0BGAHALMP0xyi5gRlR+8rLnE70A9brryYufb3QZq5ljHmY7JDFgJ5A8qlNw4O/TpJYxHF5USC3PmHlD8T5jKm/aNbdT9TLRpdqOiULQgPWt3wxdIv4kRaLUaeod9Gw3JcAbV48HMe7A69RauXFtD5m1VWtcke26iuDt7oPFUhArdSHr/Tuvyw2QdickY8DsKZzGT+A0fjZO46dwegrU4zOUqGk0BvC1uGNYFtyrJclOlymX/V6DtM8Od/7TMDW1t+szeqU56lfhRtGZDfOJsg+RZd1KYsN91zOrvTO6O8XwDrcVTZuTGwp9hOG1d4ajjWUMf9IaNy3x9ojirKiYjDLpxWDTP0XVLcU01iKS+Ftp7D/lfatiPQPb6z6vgVwudIPABWpL+3SwvmZeCcBPJOHFKJWEN0oUujt53nPgT1TMm4iKJ0Uu//EVxjktZ1Qs6USLmgk24Q9wOcM8Cv+wChzTuu4vQ8QfXBI0pUDKPbiwFGhOOq/IO3e4ap8BIcNq7X+oWvG4wsoLmC4rFjU0k4JuCf58E2WS2mpQ/spA/PYVp4z01gWPL+jXWjTk/mBMUMUCu/zcv0Y/Zz9VV2v0N3mW5rfyuqqfqtqlePmjfhCiUYYBS7vKWcNSq3UWCQYVveaecXugxuHqr7FmZsELCQvxSwLWywrzhwTU0xfjlc2k+2iXousWOgZje9uQ2/ed/7J6NUsZR7Rf1XWdDSydZSIiWR45nl7OKJJ1X6RtZe6Fey7rSulY5K+zIWXUqyi9vFQMILkHp06PALS1hxa90F5a5mADcJlsef/mcfW6kF3X0fBFep5U/zsSJfK/H36w2ZZD0znWnPVoCSu1poBgqwyTBlsmdawFTWjIUkJdYHQBdBSDkSKJS4DPq7+NMIyr0qNaErabO7UemoLkh0uvNxPBtzSrSga7Z7FvgC7iMGpi4/0qipdlF4S2LyNZsHD5XhV92ppgmT/ukE5WKxuvJrqV0wYR9Ze02P5UEdA9iJxaw+qZ6e1eXrnuOSBAPs2tmJWdUqXc+P8pCtH3J1BAZ8tIghpyr9qeBG7zsTvi8K6WBIqkG/nE295tEEQK9xmvdDsQyMSyMNZNGWuCvNHCmAqFdRXiz0liFps7X9hkTayxOrAOn5dT1lL9LyoE+XO5eAp1yx/7FHI6/NMJ+j+TH9eER2/NRJ+sFSaOrJoQX18IhEsPYJlVVG4kKGrcAUiUmMlyDfomzC4qSTfrgNRvpwJRLPljxb20HBQvQHVt5hKFkjKbiHPMX4eFjc2xAM2eqX6tVp+75Xf5HOv399T6ZNLn2cb4zbkedsZQWbeMpXPH/k2c9wcuwqOmzQQz4PG4go0wIM4/USE48AO00N/HJwt8OBO0fEnfRGUaHjfMEB4IvWZQmXz6xVeQ1nBXxED6bSsB2QdEfk+xNgvLU7FlKnu028NXE785dMfoVSIC0Vf2X4q6Xuj3P7n0Q6WL4aE1TC7Dj+p3DoRfJsStP3H1asH2N/AqryXyzh+/8/sxgHul+RGe/eMOP5KdrgaQ+W5i0mvwYQkMvVtRzpuFwCcycITU2Lo3YpeX7eySXoNWzn+ff5nAAZin1lYP8V8n7hQOqZwdPfyhMocwjuAemi9w+bEOBgNwcEIKzochZeTCEKMjYehO2mc36g2UP/hfEE2xww1bAAA="""
MANIFEST_GZIP_BASE64 = """H4sIAAAAAAACCp1dXXNcN45939+iSpEA+IFH1yST8VY2m8pUzatK47Q9mthySnayk3+/t2VZaooAce5N3hISDQIHBwDRTX06vT+9+Xz78e763f3H33+7+nTz4bf3p+vbu19O/7l6f/PP0/urd6e70/3N54/3V7cfbt6dru9uPpyu7k83768/ffz9/s3p6ub+zb9uP29ifr8//denf93cn365fvj/96e3p/vT3bYkbf9e/J+rlFJRofTNv387vbv64dXr//3x6i8fP3w43b+5vXnvCMkvhLz+4e//+PkvlDJd/7Et3GSS5szf/PdP333/Rde70+ern27/8+3t27eOTJoVk5ao7VKMZyHbP5lzS2dB767efHzz8er7Vz86+8XaX1pK33zeNL/6+dXrv3939cPN58U5yiwiV+5l1zmqpQfVWnA92iyCudI+PbrlFKa+S4iaTuFcSqqIU3Kyvdo0EUMCsnWMrPuwlQ2Ecqa8U4oBUcpM8s3/nf75GyxFLF3OIbdLl2LGXNkppcZskFh1FxvkZh9wH/Jyt5CTWTIeR1kth2XaZyQyMEzcSfdJMYBMjfs+9iaLarXvpFpii2C2sNonRSzrapJ9Uop9op0+AoCca9sHZGqxUOm6M1d2h1JrkYIwIqllsLSTP9gqJhKXfRBgk52L7tTFAHWW0vYBidnSpW7Z+7c7XIhYJYDwTlVMcuZW90mpli6t0z4pzZKS97qoW+xV204pNnRlHwdKMlm99D2OlmyGYWmldoXKTTIF0Fandag0ErPgJdk8Ayc4MYveJFJ6IUgJk3236m6fR6pFJrIz40tMt7zx5D66FQO4WylSdx5QzYDeauAdkCvJznT7WKFYlEtS90ViMQviIvvqiGIWxEX3JZFiFcScZV+7U8zmrco+tizVZCjaKSVG8pbut/y2B8nFbOi2RmqfanZHV/r5KgFhjGqy7+4m2aqGt+JxpxQDxRt/l30oriaKM++LqGpTcUrtSzzEhjWLhir72oxqUbHSznK6NvMyZROE56babXLYR3fVxitttepD2RCatSXbrPuQ1rKV4b7cTO2Q4tQMXVuF0nWzL8koizSo6GgmSoWK7LidMmsG3lmXNpNpdSdQmwVU6mmnlG5K6fug2tTKQH0nM3brxozazszc42teLqnsy0CdbJftVM3q01R2lpxdwgNuqGbad0CTgkvfR8HdRDbtvLLszY7UTRk4UnsHUKCc9xnJgvnWbuwLOU2xak1b36WaWlUFt51soGQlrraz11WOa0Dda3sV+zZ8H8WoSeC68ypSrUqDy84baG12PUqqAvXhapYZbSd3q3lTXNjrYJ/Geddvb349T+by1d8+/n7/7v3Np08P5sz5oTWc/PpiY365sSVsI73cuNkM2sjTxg3YyEaZz4htLNMn1g5trC83bqiANrbpEwv2if3lRlLMqjpblTAATNA5N+HQzgk7tWMOyRN4qICfOaFHC3jOCT4kYIhM+CkN82aeAJRbxXZOCKoZPOcEoSqgVyYMNdCfNNMPiD6aMJQUsxBNGCoJC06aMNS22g/aOWGIUQtNGJIO8uyEoQbyJU0YUgLPOWOIMcTThKEO2pbTfE7MQjxhqIDaMs3xCX7mhCHZehlo54Shc90D7ZwwlAnUdsJQVVDbmYfQc04YYgbPOWFIBfOnTBiihH2mTBjqgjGY0LwTrGd4Rh/GYCKzP8FzzjyUMb6VCUObutjONnsFtG2fCzCsHJI5lwm2s8w8BJ6zTBg698rQzglDQphXysxDhMVnmTC0teLYzglDCtYmZa6HwEq8TBhqIIOVCUO9gm3DzENgU1UnDKWO2bZOGNr6YmznhCFO2DmrUQ9h6KsGhkBtJwwRyH21ztkBi8865zKwSq1zLktgBzljCKxSm9GXYV5pRl8G7jT6MlDbCUOlYxZqE4Yy2Au2CUMNrDDaXFMLxiatzXUfaNsJQxnsANqEIarYzj7nMsU4oc81dcLO2eeLIbB66xOGWMFzGldDWGT3CUPnb6BBO+usLfiZMw8JFmW9z/7E8krX2Z8YEjTNPS+mrc71ENg76NzbZwxDOmOoYpygcy4Da02dMQRWGDpfMII1mM48BPYrOmOoYXyrM4bAbvkMof+5/eXf296705//KNf0+GVqTddkf4PJuKa2JKScrhMkgTwJkkEd2JFA0q8zJEFcOxAooTgSpNI1QxKqZ4fUQDs0TwI10Bfdk1AZ1EFdHQhFlAdKqgV0Rs4uIirojezBMhcBjZk9XCZGMZHFDS80OLKHzK3DQ83pQTOLorZobnygyMoeOLkVVISHTs4N9Ai56OyoOclFJ86ZHjqloWRBLmt2NFLJQydLQm3h8qagMUIeOlk7epDmikABTt3li4yK8LmzgyLYQ2fqaArg7AZ7Bp3KPjphES53bvSL4YLdpA6nZC4uwDMqwuVOQkss9tBJBeULdrlTUb5gnzsZtIV46JQkaKHmojOhuBC34EwK2kJc7mwo8YmLzs6oCBedDNe91U3LipqzuQdpqBZu2ZnQWks8dOaGxkhx0dlRXBS3G0poEigud24tGeaRwu5BYFu4dWdFw6wUN0ZQ+i1uZmdBPeLWnQp7pLsHqagWLjoFTcvVRWdDI7W6dSejBXR1u6KK9iPVRefGWuBBPHQSFVSEX3ei9Fv9urOgTnUzu6A1eHXRySh3VnXrCxSdzUMnFzSzNxedDcVFI7elQT3S2D0IoyLcrohgLVx0FhSdzUUn3HC3FTpBj7h1J6EXa00XN4zYQbrbFTHKWt1Hp4IH6f6NEnqv1V3urLAtxKVftKXp7o1SJ/Se0a074UjtbSECNKeHToF7s65uHkHrC02LywfsIJrdKgctDtRFJ6HEp27PntFOQF10JpT4tCxYCzyIy50Ce8TtiggtoNW9UYKv0tXlzoLa4gzOv97en96+//P69XmaxF9NcZZw/hUGMiGyJFA7xykigRwJqZ4LcEQCexIeZkyIBPEk5PNEAZFQPEtqASVUz5IJ9UVzJAgnUIfu6UAC6qCeHR5IF0JUcpXI4DFydrVAMZXJdUcFgZ09XGaBD+IBk+Hwyh4yv1yUQiI8aH4pWCERHjZJMyrCA2dOaIhlF52lgbgglzI7g1qQi05BY4Q8dMrDtRwkwkMnp46K8NApDxcfkAgPnds/qC08dHJGg52ay/+E2qK7WqDoJA+dUlFzssudFaUc9tApGY1UJpe1UMphN6fjIjx05tLRwsJDJwlKv+yh8/y1DfAgHjrPzT6ICw+dqSh6EJc7t4SIaSHJDTNFCzUXnQ3lC3HRCTtVPHSKopEqLndWtL4QN7OLgNASN7NXQc3ZXC1gj3SXclD6FQ+dqaPmLG5mfxjEQiLcbqiiObW4dSfMF8XlzgLbwq07KxqpxUVng0X4LRFadxaXOxnWwq07GcaFuqUr2l9WD51ZBbRF9Xt1NKdWF50ZDfbqoZM6WoNXF50JJb5a3NIVzWa1LkpXUAu3Y1e0dK1uy95gW7joFFSLltyrB5T4Wl6UKOD9h4vOjqKzudxZUXQ29zIpdfQgLjoZrS+an9nRorG5mb2jzURz0ckogzc3sxPK4N1FZ2dUhMudlMCDdLdnV9QjnRf9CHjD53ZFcN3Z/a4IvW/tLjofvu0LiWiLa0LwIG5XlNFOoKtb8aGspW7d+TBRhkS43CnodZCSW6KgnYD6PTvKWup2RRkW4d/Eo4WSVtcWKGtpWxSNoDn9uhPlC3XRmdHLygmcT1/Lr0cnRBffe+ZjE6KnQS56CvZ0SCgmxJOQ0clI8SR09Dqqer4QtDVsjoTMaKPdHQkE33yo582OniK7oMyoO7OHylzQ4j27sISHCdnDpcC8nT1gUoKHTC4yM4rM7EIzoUVa9rD55XtqxyZET2EOT8tcdJZ6eEL09GVlWISLTm2HJ0RP3xyEaddHJwotEjfM8uEJET1PlNPBCdHFDwaPTogufpKVDk6Inr79QocnRBfmpIMTInq+ljs6IXr66goswkNngssjdtEJj/1Y3DyihydE9Dx8TAcnRE9fSoVFuNyp6fCE6KnIQq8o2UVnInQS4KMTngR46OSK2kLczN7RGxxxK044UsVFZ4fHO8XFBR2eENHz17PywQnRxa938sEJ0dMzAunwhOirCIW/4+Whk+DLweKhUxh1anHRyWjRWFx0MnohVtzM3tAYKcXtRtC+rrjcqej1S2kuX8ADN5c7C5rNip/ZUeKrPjr58IToqQZHv4NY3czeYRHs2gKezYgbI/Ccyq870WxWPXSKwlr4HTt6y1m7GyPobKbq4gbm6ITo4gdAdHBCdPHrHTo4Ibr4tR8fnBA9Ndz98IToqROA51Ruz650eEJ0kQT44IToqVBCWau5mV3b4QkRPX9NIB2cED2xFjzecTN7R7uiTotOIB+cENHz8JEOToiefmeHxkh3M3viwxOii/YuHZwQ0fMkgA9OiOj5S/l8cEJ0cefKBydEw++1j02InnCBZjN1M3vRwxOiryJaPjwh+mpO+Hu26t/Ew1q46ISzmbrcCUeq+uhEL7HVzewVnWu8+IHbxlR7f/2eHQkM/0KZHAki8HsyjoRzm5uOvDL3rAP8K73iSejoU0fVkXD+/mE68srcszc7+ihD9yTAb3WoZwdCHxjKHigFfmAoe6gkhl9P9GDJHf3Re3ZxKfDLah4wKaO/3s8eMs+vifGhV+YutEgHX5l7dir8wFDubojpwVfmnvHNFX0WzUNngp/qIA+dwuXgK3MXlIf+ep94ISIdemXuIszQ91fIQyfD70FSdXGBQos8dLLCL+519yAoX5DLnfADAOxyZzv6ytwzLvD33Wjh1HTolbnnSIWfGmU3qXdYCw+diVCAs8udDdbC5U6mg6/MPWtB6NtV7KETf+RTPHSef2dHh16Zez4I/C6y+NyJRqp46OSKhpmIG2awFm5m53LwlbmLkjEdfGXuAhdocSA+d8Ja6MKpdOiVuYuDoE+NFpc7O5rNiotO+NG+4nIn/GBIcTM7/jib2xHBb44UF50FLdeKi05Cw6y4TVFBnyYuHjpJUBHVzewKP4vmcicM8Oo26wSLYLdEQbvUKm4Zj6Kz+nWnHnxl7sIjcMvvZvaCvhNaPXSmBJvT7Yoy/L69e5GU+8FX5p5FwABvbt3JaBnf3Myu6eArc88iEoqL5mZ2/MU9t+6UdPCVuWd0VpR+m4tOOI80Xdwc8KFX5i5yaj/4ytwFX8DPormZXeDrORedcNHY3bqzoTcH3eXOrAdfmbvQAn73z6870d6su+hkPvjK3EV9gXKnutyp9eArcxceQaGlqxslOvTK3MU0AX3wVGXhkWOvzF3cd6LNhLqZndLBV+YucAG/PujWnfBz/+rXnehc4wzOb//6/asfv/zxoi9/aunNxzcfr7b/Zg2Dnhbz498K9xbT5WIiXi7mYfHj3yD0FsvlYu3rxeVy8RYwy8X1cnF+/MON3uI2mK6vD9gvF4uuF+vl4p7WB8yDCzfjrVcPPqTHvw7orh6c2B7/Uru7mkd8rA+ZBzdyXXsmD36sgbXz4EitgQXbCKi12/Pgyg0l69WDL1NdBxil0YKyXj34smlfrx582fP6lDT4MgU2ocGXEuk9+LJHq0dfljViafBl74HswZelBhYcfUlr2Tz4Muvagjz4UgJ880iuvI5LHnz59W/DuqtHeg28w4MvOYgGrmM0BJq0EYNrhuDBlxzwIA++lCAaZEyTgedl5NggLmWMS1njW0ZfRpqMHBtEgwy+zC3QZIxLXWNQ2phJ1jwoY1ymQJPRlyUoCEZfBnFZRo7N61OWwZe5rzFYRl+WNb7L6EtZW7CMcZmC1WPhE+CkDL7MASOXwZc1KKrK4EtNQb02+LIHGKyDLysFq1/E5dqCdfClBJVSlbF2DFYPvqSghqiDL0tg7zpWsZEF++j5QPboS11HQxt9mdas2QZfphSU6oMvS1lbsA2+bJFsGeuqQO8yNlyBTQZftqBab4Mva1vn4jbmSw1sMviy5DVD9NGXQfXY88iDa5v0F7VPsJrHlm6dG/rgyxxgsI9xmdax00dftkD2WMdKIHvk2MjeYx0b5EsdfFl1zcg6+pLWiNUxXwbMpoMvaxCXOnJsUPXq6EsKTllHewenHHxJAWtqH2M+0Hvk2CAuz678/tUPr3/6koq/gOrnV6///p133fO8+rE8cVfT5er2+Iff3dV8uZoeS013tVyu1sc/Qu6uLper+6NJ3NX1cnV5DHl3dRtO+djAuKv75WrOgd46asKBdwZncrh88GaRQJc8uLNSYMQ8+PMrAfnLXzg0MHoePNooQEt+4dLoqINPtUZH7aPukTKDV7kFytDg1cxR0A1e1RIYkgav5hpJH7xaorijwasllD54tXAkffDqloiC5WOk5iBSafBqK5F0HTkmYrs0QiyQzoNXWQND8uBVpkj64NVMkfTBq7UF4cFlZPYgx/CLWI2ktzGaAtLjwatSAojxyMAS6C4jA3MAAhm9mgO7y4uEGi0fGbhFygxe7REDy8jAHOXrwasapWB5wcDRUUcGliC0ZfBqlSA8ShpJL4BYGbyaelSbDF7tHFimDF6V6Khl9CpHuo9e1cCrZfBqL5Hug1cpBV4tY7GUImV0zHxRlTd4tUXEUcfatwdMUMe8GiXKOng1RRCrg1dziZQpY5Uf2L0OXq0ahEdtY3gEPFP7iMjAq3X0qkbV+FgtRVm7jV6N8N5GBo4Q2QavaoqUGWM1BV5tL2rgyDJ1LH8iZUYGjnJT6yPEIsvoWHYGeO8vGDg4ah+82nqge6ex7AwA3EcG1oBn+uDVFPF7H7wqHPWTIwNHtVhvY/YI3NRHr0YA7jpmj0AZHbxK0VF19GqLltMYfIFXdayWJFo+MjAHltHBqykKPq0jCKJrgsGrKWIx7SMiI2XGaqkEdj879debu19u7z79+uc1ffkzlzzc/vxw8/nb27dvrdulaWcebOXsJGtnyT3eydbOPhjc2SnWThms4+wsprY1xTuraaEh6J2dzdpZE6Btt3aqAP5UU9uhWPGQYIKoI+pmE0UVwp8JIyoAGLKJo5oBn2YTSMoA7LOJpNyRs5pQSgKETLaxRMhZux2oGdhqoqk14Kxko6kBQKRsQwL5VBNNUpFPZds5QJyT2GcFnEMmmliAoCMTTVoY2GqiSTuy1URTQzBMJpokAZ/KJpqKIltNNAnC32yiqSUATWznuAQEHZtoqgVAE5to6op8qp3mGPlUm5sagGE20ZQUcY6JJs6Ac8QulxCFxeamBlhYTDQlBjAsbKcrIOhE7HSFfKqJJlIgSYqd6ZDiUBw0IVvtsgnyq4mmjiTJkmyCASqYYlffDXBOITtegcgpTt2EfKqT6RAzmWgab5m9rdWmNaTsN9GUBTmrgyYAiEVt+CPdhp3pEL9WE01cATNVG00IhquJJkI6yGqiqSBtTrWrcIT9a7XLS8TCJpokIwp3u9BDFFa7dwUs3Ew0KVI3NRNNDanCm103IQTTTDQxUjc1sZsVIOiaUzchPb7NTUg72JqdX4F4bSaacgLitdk9XQbO2u2ergBA7E5PB0Ci2xdNSFHbTTQ1JHK6XYULYOFuZzqka+7VDnWAJbqd6TKisI2miihs93QKBJ0mOzUjl1x2pkNyjtpVuCAK29wEndVEEyORo3amQ8oQtdGEXGlos9MV4hwTTYQEnapdSwCf+gCmP25//Xj3+XT3+frD089xr/+oDx5Gbp5yIKQi6ZoCIQ0JSQ6ElI7eli+EZOT6sETHQVrJGghhxDstEKKk4M36QggResm+ECLQpXkKVRH06n2FWeTiKEegVSTd5gi13DN6N79CHHSiCLcVucbLEXAF6V9zhNyEVF+5h+zU0Sv9hZSOTNMowm5FameKsNsIvvNf0Rx0rx1htyLlEEXYTdCJIuwWpM+lELsEjwpW1oXmMiHtIl0PaejphM4SlrlV0LHCSpdE6IRhZRdkKs4RdlNv6NxhWUJldASxkoLcvXGE3YrUhBxiF7lB5x7mAELHFavciIxaJMRugYcYS10qOs9YYRfShUMfJXTKseQ6QQceK+tCU4gaMmZFxyArKch9gkTY1d7R4ciyIqvonGSFOiSOSoTdgvRoJcIul44OUlbWZUJnKitP54aOV5a1N6OTllVMQ6OeFvaMGZ2/rCpVaMQW9moVnsqsshrCUjXCbkca8hr2akhPUsNeDRo5hTUDUr/UkHch64b1LjQdCrGLfDWththFrsNqhN2E6NIi7HbJ6DhoWXvDk6HlbQWhQ6JlD1vQedEyN8Kjo2VMd3SKtMwBgg6UllUQobOllRTknqFp2MNmdOK0rBkEHT4tmUHQOdSy41N0JLW8CiV0OrW8ZE7ooGolBfqucg27z46Or1Z2oYxOspY+auhQa8UMouh8a8V1BI+6lvzC6NRreXee0QHYMo46Ogtb1ZhINGpc7xI6IVvpgnx7RluY1Ridmy3vVAkdoa1i2jzR/wO0/FJ/qPEAAA=="""

runtime_dir = Path("/kaggle/working/aide_original_full_commfor_runtime")
runtime_dir.mkdir(parents=True, exist_ok=True)
train_runtime_path = runtime_dir / "train_aide_original_full_tiny_commfor_kaggle.py"
test_runtime_path = runtime_dir / "test_aide_original_full_commfor_exact_kaggle.py"
manifest_path = runtime_dir / "selected_samples.csv"

train_runtime_path.write_bytes(gzip.decompress(base64.b64decode(TRAIN_RUNTIME_GZIP_BASE64)))
test_runtime_path.write_bytes(gzip.decompress(base64.b64decode(TEST_RUNTIME_GZIP_BASE64)))
manifest_path.write_bytes(gzip.decompress(base64.b64decode(MANIFEST_GZIP_BASE64)))

sys.path.insert(0, str(runtime_dir))
spec = importlib.util.spec_from_file_location(
    "aide_original_full_commfor_runner", test_runtime_path
)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

CONFIG["manifest_path"] = str(manifest_path)
print("Runtime:", test_runtime_path)
print("Embedded exact manifest:", manifest_path)
print("Full AIDE CommFor inference runner imported successfully.")

In [ ]:
result = runner.run_inference(CONFIG)
result["experiment"]

In [ ]:
import json
import pandas as pd
from IPython.display import FileLink, display

run_dir = Path(CONFIG["output_root"])

print("CommFor overall: 100 real + 900 fake unique samples")
display(pd.DataFrame([json.loads((run_dir / "metrics/commfor_overall.json").read_text())]))

print("CommFor per generator: mỗi hàng = cùng 100 real + 100 fake của generator đó (%)")
per_generator = pd.read_csv(run_dir / "metrics/commfor_generator_metrics.csv")
metric_columns = [
    "accuracy", "balanced_accuracy", "real_recall", "fake_recall",
    "fake_precision", "fake_f1", "roc_auc", "average_precision",
]
shown = per_generator.copy()
for column in metric_columns:
    shown[column] = (100 * shown[column]).round(2)
display(shown)

print("Macro summary")
display(pd.DataFrame([json.loads((run_dir / "metrics/commfor_macro_summary.json").read_text())]))

print("Download:")
display(FileLink("/kaggle/working/aide_original_full_commfor_exact_eval.zip"))

## Output quan trọng

- `metrics/commfor_overall.json`: metric trên 1.000 ảnh unique (lưu ý tỷ lệ lớp 100 real / 900 fake).
- `metrics/commfor_macro_summary.json`: macro trung bình qua 9 cohort cân bằng theo generator.
- `metrics/commfor_generator_metrics.csv`: chi tiết từng generator, mỗi hàng gồm 100 shared real + 100 fake.
- `predictions/commfor_predictions.csv`: xác suất và dự đoán từng ảnh.
- `dataset/commfor_manifest_used.csv`: manifest chính xác đã sử dụng.
- `provenance/checkpoint.json`: checksum và metadata checkpoint.
- `aide_original_full_commfor_exact_eval.zip`: gói kết quả tải về.